# 01 — Run Tests Locally

Run the full test suite from your local machine. No Databricks cluster needed.

This demonstrates the recommended workflow: **develop and test locally, deploy to Databricks with confidence.**

## Setup

Make sure you've installed the dependencies:
```bash
python -m venv .venv
source .venv/bin/activate
pip install -r requirements.txt
```

> **Note:** `databricks-connect` bundles PySpark. If you have standalone `pyspark` installed, uninstall it first to avoid conflicts.

## Step 1: Look at the code we're testing

Let's peek at `baseball_stats.py` — the functions under test.

In [ ]:
# Quick look at the functions we wrote
from baseball_stats import batting_average, slugging_percentage, classify_hitter

# Try them out manually first
print("Batting average (150 H / 500 AB):", batting_average(150, 500))
print("Slugging (80 1B, 20 2B, 5 3B, 15 HR / 400 AB):", slugging_percentage(80, 20, 5, 15, 400))
print("Classification for .320:", classify_hitter(0.320))
print("Classification for .210:", classify_hitter(0.210))

## Step 2: Run pure Python tests

These tests validate the math functions — no Spark involved. They run in milliseconds.

In [ ]:
import pytest
import sys

# Run ONLY the pure Python tests
# -v = verbose (show each test name and result)
# -p no:cacheprovider = don't write .pytest_cache (cleaner in notebooks)
retcode = pytest.main([
    "tests/test_pure_python.py",
    "-v",
    "-p", "no:cacheprovider"
])
print(f"\nReturn code: {retcode} ({'PASSED' if retcode == 0 else 'FAILED'})")

## Step 3: Run Spark transformation tests

These tests create a local SparkSession and validate DataFrame operations. 
The first run takes ~2-3 seconds to start Spark; after that, the session is reused.

In [ ]:
# Run ONLY the Spark tests
retcode = pytest.main([
    "tests/test_spark_transforms.py",
    "-v",
    "-p", "no:cacheprovider"
])
print(f"\nReturn code: {retcode} ({'PASSED' if retcode == 0 else 'FAILED'})")

## Step 4: Run the full suite

In practice, you run everything at once. This is what CI/CD would do on every commit.

In [ ]:
# Run ALL tests in the tests/ directory
retcode = pytest.main([
    "tests/",
    "-v",
    "-p", "no:cacheprovider"
])
assert retcode == 0, "Some tests failed — check the output above."

## What Just Happened?

1. **pytest discovered** all files matching `test_*.py` in the `tests/` folder
2. **conftest.py** created a local SparkSession (once) and sample DataFrames
3. **Pure Python tests** ran instantly — just math
4. **Spark tests** created DataFrames, ran transformations, checked results
5. **All tests** used fake data we control — no production dependencies

You can run this exact same `pytest tests/ -v` command:
- From your terminal
- In a Databricks notebook (next notebook)
- In GitHub Actions / CI pipeline
- Via Databricks Connect against a remote cluster

The tests don't change. Only the Spark backend does.